# Look at mapping breadth and depth of 100 x metagenomes to singleclust genes

...find more candidate genes

In [1]:
import polars as pl
import glob
import os
import screed
import csv
import screed

In [4]:
seq_to_species_df = (
    pl.read_csv('../outputs.mapping/cds-min50-singleclust/species_to_genes.csv')
    .rename({ 'ident': 'gene' })
    .select(['gene', 'species'])
)
seq_to_species_df

gene,species
str,str
"""AtH2023_ERR1135319_MAG02_1_7""","""s__Bariatricus sp004560705"""
"""AtH2023_ERR1135319_MAG02_1_10""","""s__Bariatricus sp004560705"""
"""AtH2023_ERR1135319_MAG02_1_13""","""s__Bariatricus sp004560705"""
"""AtH2023_ERR1135319_MAG02_1_23""","""s__Bariatricus sp004560705"""
"""AtH2023_ERR1135319_MAG02_1_27""","""s__Bariatricus sp004560705"""
…,…
"""JALFGL010000073.1_2""","""s__UBA2868 sp004552595"""
"""CAMTHS010000005.1_1""","""s__UBA2868 sp004552595"""
"""CAMTHS010000062.1_11""","""s__UBA2868 sp004552595"""


In [5]:
seq_to_species_df.group_by('species').agg(pl.len()).sort('len')

species,len
str,u32
"""s__JAFBIX01 sp021531895""",1
"""s__Floccifex porci""",17
"""s__Mogibacterium_A kristiansen…",21
"""s__Fimisoma sp002320005""",40
"""s__Lactobacillus amylovorus""",182
…,…
"""s__Ornithospirochaeta sp022785…",770
"""s__Prevotella sp000434975""",851
"""s__Cryptobacteroides sp9005469…",928


## Read in summarized metag x species singleclust depth reports

This file contains depth-of-mapping information for all our genes.

In [11]:
depth_df = (
    pl.read_csv('../outputs.mapping/rand100-cds3.min50.depth.csv')
    .join(seq_to_species_df, on=['gene'])
)

In [12]:
depth_df

metag,gene,len,hits,breadth,depth_all,depth_cov,species
str,str,i64,i64,f64,f64,f64,str
"""ERR3211766""","""AtH2023_SRR5976187_MAG13_142_1""",843,843,1.0,9.85172,9.85172,"""s__Prevotella sp002251295"""
"""ERR3211766""","""JAJXEN010000047.1_8""",2916,735,0.252058,0.363169,1.440816,"""s__Prevotella sp000434975"""
"""ERR3211766""","""AtH2023_SRR8960514_MAG08_242_7""",567,0,0.0,0.0,NaN,"""s__Colivicinus sp002299675"""
"""ERR3211766""","""JAQYGO010000082.1_2""",1488,302,0.202957,0.202957,1.0,"""s__UBA2868 sp004552595"""
"""ERR3211766""","""AtH2023_SRR5241533_MAG03_126_2""",2151,2151,1.0,15.751278,15.751278,"""s__Sodaliphilus sp004557565"""
…,…,…,…,…,…,…,…
"""SRR11125751""","""AtH2023_SRR8960514_MAG08_157_1…",4413,3820,0.865624,2.273057,2.625916,"""s__Colivicinus sp002299675"""
"""SRR11125751""","""CAVEFX010000252.1_11""",1239,1171,0.945117,4.621469,4.889838,"""s__Sodaliphilus sp004557565"""
"""SRR11125751""","""JAHAYU010000085.1_10""",2349,2255,0.959983,2.862069,2.981375,"""s__Cryptobacteroides sp9005469…"


In [13]:
depth_df.filter(pl.col('depth_all').is_not_nan()).sort(by='depth_all', descending=True).filter(pl.col('depth_all') > 0.0)

metag,gene,len,hits,breadth,depth_all,depth_cov,species
str,str,i64,i64,f64,f64,f64,str
"""SRR14369134""","""SFFG01000401.1_1""",702,702,1.0,19357.985755,19357.985755,"""s__Sodaliphilus sp004557565"""
"""SRR17241521""","""AtH2023_SRR11183406_MAG1_196_4…",324,324,1.0,16832.611111,16832.611111,"""s__Lactobacillus amylovorus"""
"""SRR11489750""","""SFFG01000401.1_1""",702,702,1.0,15167.830484,15167.830484,"""s__Sodaliphilus sp004557565"""
"""SRR10209683""","""AtH2023_SRR8960486_MAG03_96_2""",375,375,1.0,15122.426667,15122.426667,"""s__Mogibacterium_A kristiansen…"
"""SRR11489783""","""AtH2023_SRR8960486_MAG03_96_2""",375,375,1.0,11142.842667,11142.842667,"""s__Mogibacterium_A kristiansen…"
…,…,…,…,…,…,…,…
"""SRR8960492""","""AtH2023_ERR3211862_MAG2_81_1""",4197,2,0.000477,0.000477,1.0,"""s__Prevotella sp002251295"""
"""SRR8960379""","""HF986301.1_56""",2430,1,0.000412,0.000412,1.0,"""s__Cryptobacteroides sp0004326…"
"""SRR11124923""","""CAVAXB010000020.1_12""",2745,1,0.000364,0.000364,1.0,"""s__Sodaliphilus sp004557565"""


## Summarize our mapping breadth results across all the metagenomes

This is for figuring out which genes to manually curate.

In [14]:
# require 10% of each gene to be covered by at least one read
BREADTH_CUTOFF = 0.1

In [15]:
# aggregate across all metagenomes;
# calculate fraction of metagenomes for which gene mapping exceeds our breadth cutoff
agg_df = depth_df.group_by(['species', 'gene']).agg(
    # get fraction of columns where breadth is greater than cutoff as 'f'
    ((pl.col("breadth") >= BREADTH_CUTOFF).sum() / pl.col("breadth").len()).alias("f"),
#    (pl.col("depth").filter(pl.col("depth").is_not_nan()).mean()),
)
agg_df

species,gene,f
str,str,f64
"""s__Cryptobacteroides sp9005469…","""AtH2023_SRR17241489_MAG21_95_2""",0.66
"""s__Cryptobacteroides sp9005469…","""AtH2023_ERR8314740_MAG4_323_4""",0.82
"""s__Sodaliphilus sp004557565""","""AtH2023_ERR3211826_MAG3_143_3""",0.92
"""s__Prevotella sp002251295""","""AtH2023_SRR5976187_MAG13_152_1…",0.87
"""s__Colivicinus sp002299675""","""CAMFSL010000016.1_44""",0.68
…,…,…
"""s__Cryptobacteroides sp0004349…","""JAXXGF010000032.1_4""",0.86
"""s__Cryptobacteroides sp9005469…","""AtH2023_ERR3211746_MAG11_207_1""",0.86
"""s__Sodaliphilus sp004557565""","""AtH2023_SRR7182061_MAG3_144_1""",0.71


## Investigate specific species


In [19]:

SPECIES_INVESTIGATE = {
    's__Floccifex porci': 'BAD',
    's__Bariatricus sp004560705': 'BAD',
    's__Cryptobacteroides sp034089285': 'BAD',
    's__UBA2868 sp004552595': 'BAD',
    's__Mogibacterium_A kristiansenii': 'BAD',
    's__JAFBIX01 sp021531895': 'WTF',                      # no single copy gene yet.
}

In [24]:
ofp = open('../candidate-genes-for-review.csv', 'w', newline='')
w = csv.writer(ofp)

# print information out by species
for species in sorted(agg_df['species'].unique()):
    if species not in SPECIES_INVESTIGATE:
        continue
        
    print(species)
    foo_df = agg_df.filter(pl.col("species") == species)
    foo_df = foo_df.sort(by='f', descending=True).filter(pl.col('f') > 0.0)
    print(foo_df)

    for name in foo_df.head(50)['gene'].to_list():
        w.writerow(['0', '0', species, '(not reviewed)', name, '(not reviewed)'])

ofp.close()
    

s__Bariatricus sp004560705
shape: (461, 3)
┌────────────────────────────┬─────────────────────────────────┬──────┐
│ species                    ┆ gene                            ┆ f    │
│ ---                        ┆ ---                             ┆ ---  │
│ str                        ┆ str                             ┆ f64  │
╞════════════════════════════╪═════════════════════════════════╪══════╡
│ s__Bariatricus sp004560705 ┆ CAMEGG010000087.1_1             ┆ 1.0  │
│ s__Bariatricus sp004560705 ┆ AtH2023_ERR1135319_MAG02_256_1… ┆ 0.99 │
│ s__Bariatricus sp004560705 ┆ AtH2023_ERR1135319_MAG02_116_2… ┆ 0.99 │
│ s__Bariatricus sp004560705 ┆ AtH2023_ERR1135319_MAG02_177_1… ┆ 0.99 │
│ s__Bariatricus sp004560705 ┆ AtH2023_ERR1135319_MAG02_177_1… ┆ 0.99 │
│ …                          ┆ …                               ┆ …    │
│ s__Bariatricus sp004560705 ┆ CAMEYJ010000133.1_7             ┆ 0.48 │
│ s__Bariatricus sp004560705 ┆ AtH2023_SRR17241646_MAG19_72_1… ┆ 0.48 │
│ s__Bariatricus sp00